In [0]:
# CAMADA GOLD
# Databricks notebook source
# COMMAND ----------
# 1. IMPORTS E CONFIGURAÇÃO INICIAL
from pyspark.sql.functions import col, count, sum, avg, round, md5, concat_ws, when

# Garante a criação do schema gold
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp_eng_dados.gold")

# Leitura das fontes tratadas na camada Silver
df_silver_egressos = spark.read.table("mvp_eng_dados.silver.dataset_egressos")
df_silver_cursos = spark.read.table("mvp_eng_dados.silver.dataset_cursos")

# COMMAND ----------
# 2. MODELAGEM DIMENSIONAL (ESQUEMA ESTRELA)

# 2.1 Dimensão: Egresso
dim_egresso = (
    df_silver_egressos
    .select(
        col("id_egresso"),
        col("idade_egresso"),
        col("uf_residencia"),
        col("bolsista_graduacao")
    )
    .dropDuplicates(["id_egresso"])
)
dim_egresso.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_eng_dados.gold.dim_egresso")

# 2.2 Dimensão: Curso (Poputada via silver.cursos)
# Associa o catálogo de cursos (silver.cursos) com atributos de modalidade e área provenientes dos egressos
df_atributos_curso = (
    df_silver_egressos
    .select("id_curso", "area_atuacao", "modalidade_graduacao")
    .dropDuplicates(["id_curso"])
)

dim_curso = (
    df_silver_cursos
    .join(df_atributos_curso, on="id_curso", how="left")
    .select(
        col("id_curso"),
        col("nome_curso"),
        col("area_conhecimento"),
        col("duracao_semestres"),
        col("mensalidade_base"),
        col("modalidade_graduacao")
    )
    .dropDuplicates(["id_curso"])
)
dim_curso.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_eng_dados.gold.dim_curso")

# 2.3 Dimensão: Perfil de Emprego
dim_emprego = (
    df_silver_egressos
    .select("nivel_cargo", "tipo_empresa")
    .dropDuplicates()
    .withColumn("id_emprego", md5(concat_ws("||", col("nivel_cargo"), col("tipo_empresa"))))
    .select("id_emprego", "nivel_cargo", "tipo_empresa")
)
dim_emprego.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_eng_dados.gold.dim_emprego")

# 2.4 Tabela Fato: Egressos
fato_egressos = (
    df_silver_egressos
    .withColumn("id_emprego", md5(concat_ws("||", col("nivel_cargo"), col("tipo_empresa"))))
    .withColumn("categoria_nps", 
        when(col("satisfacao_graduacao_nps") >= 9, "Promotor")
        .when(col("satisfacao_graduacao_nps") >= 7, "Neutro")
        .otherwise("Detrator")
    )
    .withColumn("faixa_meses_formacao",
        when(col("meses_desde_formacao") <= 12, "0-12m")
        .when(col("meses_desde_formacao") <= 24, "13-24m")
        .when(col("meses_desde_formacao") <= 36, "25-36m")
        .when(col("meses_desde_formacao") <= 48, "37-48m")
        .when(col("meses_desde_formacao") <= 60, "49-60m")
        .otherwise(">60m")
    )
    .select(
        # Chaves Estrangeiras (FKs)
        col("id_egresso"), 
        col("id_curso"), 
        col("id_emprego"), 
        col("dt_ultimo_contato"),

        # Métricas / Fatos
        col("renda_mensal_estimada"),
        col("satisfacao_graduacao_nps"), 
        col("categoria_nps"),
        col("engajamento_alumni_score"), 
        col("potencial_matricula_pos"), 
        col("meses_desde_formacao"),
        col("faixa_meses_formacao")
    )
)
fato_egressos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_eng_dados.gold.fato_egressos")

# COMMAND ----------
# 3. DATA MARTS DE KPIS (ATENDENDO ÀS 11 PERGUNTAS)

# 3.1 KPI Conversão Pós por Curso e Área (Perguntas 1 e 9)
df_gold_pos = (
    df_silver_egressos.join(df_silver_cursos, "id_curso")
    .groupBy("nome_curso", "area_conhecimento", "modalidade_graduacao")
    .agg(
        count("id_egresso").alias("total_egressos"),
        sum("potencial_matricula_pos").alias("total_potencial_pos"),
        round(avg("satisfacao_graduacao_nps"), 2).alias("media_nps")
    )
    .withColumn("taxa_potencial_pos_pct", round((col("total_potencial_pos") / col("total_egressos")) * 100, 2))
)
df_gold_pos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_eng_dados.gold.kpi_potencial_pos_graduacao")

# 3.2 KPI Tempo de Formação e Bolsistas (Perguntas 2 e 3)
df_gold_tempo_bolsa = (
    fato_egressos.join(dim_egresso, "id_egresso")
    .withColumn(
        "faixa_meses_formacao",
        when(col("meses_desde_formacao") <= 12, "0 a 12 meses")
        .when(col("meses_desde_formacao") <= 24, "13 a 24 meses")
        .when(col("meses_desde_formacao") <= 36, "25 a 36 meses")
        .otherwise("Mais de 36 meses")
    )
    .groupBy("faixa_meses_formacao", "bolsista_graduacao")
    .agg(
        count("id_egresso").alias("total_egressos"),
        sum("potencial_matricula_pos").alias("total_potencial_pos")
    )
    .withColumn("taxa_potencial_pos_pct", round((col("total_potencial_pos") / col("total_egressos")) * 100, 2))
)
df_gold_tempo_bolsa.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_eng_dados.gold.kpi_conversao_tempo_bolsa")

# 3.3 KPI Empregabilidade, ROI e Setores (Perguntas 4, 5 e 6)
df_gold_emprego = (
    df_silver_egressos.join(df_silver_cursos, "id_curso")
    .groupBy("nome_curso", "modalidade_graduacao", "nivel_cargo", "tipo_empresa", "mensalidade_base")
    .agg(
        count("id_egresso").alias("qtd_egressos"),
        round(avg("renda_mensal_estimada"), 2).alias("renda_media")
    )
    .withColumn("razao_roi_renda_mensalidade", round(col("renda_media") / col("mensalidade_base"), 2))
)
df_gold_emprego.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_eng_dados.gold.kpi_empregabilidade_roi_curso")

# 3.4 KPI NPS, Engajamento Alumni e Propensão Pós (Perguntas 7, 8 e 9)
df_gold_nps = (
    fato_egressos.join(dim_curso, "id_curso")
    .withColumn(
        "categoria_nps",
        when(col("satisfacao_graduacao_nps") >= 9, "Promotor")
        .when(col("satisfacao_graduacao_nps") >= 7, "Neutro")
        .otherwise("Detrator")
    )
    .groupBy("categoria_nps", "satisfacao_graduacao_nps", "modalidade_graduacao")
    .agg(
        count("id_egresso").alias("total_egressos"),
        round(avg("engajamento_alumni_score"), 2).alias("media_engajamento_alumni"),
        sum("potencial_matricula_pos").alias("total_potencial_pos")
    )
    .withColumn("taxa_potencial_pos_pct", round((col("total_potencial_pos") / col("total_egressos")) * 100, 2))
)
df_gold_nps.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_eng_dados.gold.kpi_nps_engajamento_alumni")

# 3.5 KPI Geográfico e Mercado Regional (Perguntas 10 e 11)
df_gold_perfil_uf = (
    df_silver_egressos
    .groupBy("uf_residencia", "area_atuacao", "nivel_cargo", "tipo_empresa")
    .agg(
        count("id_egresso").alias("qtd_egressos"),
        sum("potencial_matricula_pos").alias("total_potencial_pos"),
        round(avg("renda_mensal_estimada"), 2).alias("renda_media"),
        round(avg("meses_desde_formacao"), 1).alias("media_meses_formado")
    )
    .withColumn("taxa_potencial_pos_pct", round((col("total_potencial_pos") / col("qtd_egressos")) * 100, 2))
)
df_gold_perfil_uf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("mvp_eng_dados.gold.kpi_perfil_profissional_uf")

# COMMAND ----------
# 4. OTIMIZAÇÃO E COMPACTAÇÃO DE LEITURA (Z-ORDERING)

# Tabela Fato
spark.sql("OPTIMIZE mvp_eng_dados.gold.fato_egressos ZORDER BY (id_curso, id_egresso)")

# Tabelas Dimensionais
spark.sql("OPTIMIZE mvp_eng_dados.gold.dim_curso ZORDER BY (id_curso)")
spark.sql("OPTIMIZE mvp_eng_dados.gold.dim_egresso ZORDER BY (id_egresso)")

# Data Marts / KPIs
spark.sql("OPTIMIZE mvp_eng_dados.gold.kpi_potencial_pos_graduacao ZORDER BY (nome_curso)")
spark.sql("OPTIMIZE mvp_eng_dados.gold.kpi_conversao_tempo_bolsa ZORDER BY (faixa_meses_formacao)")
spark.sql("OPTIMIZE mvp_eng_dados.gold.kpi_empregabilidade_roi_curso ZORDER BY (nome_curso)")
spark.sql("OPTIMIZE mvp_eng_dados.gold.kpi_nps_engajamento_alumni ZORDER BY (categoria_nps)")
spark.sql("OPTIMIZE mvp_eng_dados.gold.kpi_perfil_profissional_uf ZORDER BY (uf_residencia)")